# 05 — Gold queries (samples)

Use these as starting points; pin to Power BI via the SQL analytics endpoint of `ObservabilityLH`.

In [3]:
import json, os
from pyspark.sql import functions as F

# When run inside Fabric, the notebook resource folder contains config.json.
# Fabric exposes notebook-attached files via mssparkutils / notebookutils.
try:
    import notebookutils  # type: ignore
    cfg_path = notebookutils.nbResPath + '/builtin/config.json'
    if not os.path.exists(cfg_path):
        # Fallback: lakehouse Files/config.json
        cfg_path = '/lakehouse/default/Files/config.json'
except Exception:
    cfg_path = './config.json'

with open(cfg_path, 'r', encoding='utf-8') as f:
    CFG = json.load(f)

OBS_WS  = CFG['observability_workspace_name']
OBS_LH  = CFG['observability_lakehouse_name']
TBL     = CFG['tables']
API     = CFG['fabric_api']
# monitored_workspaces is a list of workspace display names (strings).
# Backwards-compat: also accept the old [{workspace_name: ...}] shape.
_raw_mon = CFG['monitored_workspaces']
MONITOR = [m if isinstance(m, str) else m['workspace_name'] for m in _raw_mon]
INGEST  = CFG['ingestion']
print(f'Observability workspace : {OBS_WS}')
print(f'Observability lakehouse : {OBS_LH}')
print(f'Monitored workspaces    : {MONITOR}')

StatementMeta(, 299074df-66c1-4009-bfa8-22dd04ac8c6a, 3, Finished, Available, Finished, False)

Observability workspace : WS_OnelakeObservability
Observability lakehouse : lh_OnelakeObservability
Monitored workspaces    : ['WS_SagarFabric01', 'WS_SagarFabric03']


**Data captured by Onelake diagnostic is granular i.e. a row is generated for every activity on the delta table/file. So there may be more than 1 record for the same operation on a given delta table for delta log, each parquet file touched etc.**
**Hence, creating an aggregate table to capture 1 record for each unique operation on a delta table/file**

In [10]:
gold_df = spark.sql("""
    SELECT
        event_date, event_emitting_workspace, executingUPN, executingPrincipalType, executingPrincipalId,
        callerIPAddress, originatingApp, accessStartTime, accessEndTime, operationName, operationCategory,
        serviceEndpoint, httpStatusCode, bytes, isShortcut,
        shortcut_consumer_workspace, shortcut_consumer_lakehouse, 
        regexp_extract(shortcut_path_consumer, '(?:^|/)((?:Tables|Files)/[^/]+)', 1) AS shortcut_consumer_path,
        target_type, target_workspace_id,
        target_workspace_name, target_item_id, target_item_name, target_path, target_connection_id,
        target_location, target_subpath
    FROM dbo.silver_onelake_enriched_access
    GROUP BY
        event_date, event_emitting_workspace, executingUPN, executingPrincipalType, executingPrincipalId,
        callerIPAddress, originatingApp, accessStartTime, accessEndTime, operationName, operationCategory,
        serviceEndpoint, httpStatusCode, bytes, isShortcut,
        shortcut_consumer_workspace, shortcut_consumer_lakehouse, target_type, target_workspace_id,
        target_workspace_name, target_item_id, target_item_name, target_path, target_connection_id,
        target_location, target_subpath,
        regexp_extract(shortcut_path_consumer, '(?:^|/)((?:Tables|Files)/[^/]+)', 1)
""")

StatementMeta(, f0d37969-0563-46f6-aa9a-a39c1d88b281, 12, Finished, Available, Finished, False)

In [9]:
display(gold_df)

StatementMeta(, f0d37969-0563-46f6-aa9a-a39c1d88b281, 11, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 864da38b-2d88-49c0-81b3-9e033cb2d626)

In [11]:
gold_df.write.format('delta') \
    .mode('overwrite') \
    .option('overwriteSchema', 'true') \
    .saveAsTable('dbo.gold_onelake_enriched_access')

StatementMeta(, f0d37969-0563-46f6-aa9a-a39c1d88b281, 13, Finished, Available, Finished, False)

## Top users copying shortcut-backed data in the last 24h

df = spark.sql(f'''
SELECT executingUPN, target_type,
       COUNT(*)              AS ops,
       SUM(COALESCE(bytes,0)) AS bytes_read,
       COLLECT_SET(originatingApp)        AS apps,
       COLLECT_SET(callerIPAddress)       AS ips,
       COLLECT_SET(shortcut_path_consumer) AS shortcut_paths
FROM {TBL['silver']}
WHERE accessStartTime > current_timestamp() - INTERVAL 1 DAY
  AND operationCategory = 'Read'
GROUP BY executingUPN, target_type
ORDER BY ops DESC
''')

In [15]:
df = spark.sql(f'''
SELECT executingUPN, target_type,
       COUNT(*)              AS ops,
       SUM(COALESCE(bytes,0)) AS bytes_read,
       COLLECT_SET(originatingApp)        AS apps,
       COLLECT_SET(callerIPAddress)       AS ips,
       COLLECT_SET(target_workspace_name) AS target_workspace_name,
       COLLECT_SET(target_item_name) AS target_item_name,
       COLLECT_SET(target_path) AS target_path
FROM gold_onelake_enriched_access
WHERE accessStartTime > current_timestamp() - INTERVAL 1 DAY
  AND operationCategory = 'Read'
GROUP BY executingUPN, target_type
ORDER BY ops DESC
''')
display(df)

StatementMeta(, f0d37969-0563-46f6-aa9a-a39c1d88b281, 17, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4e503150-1d1d-458d-9263-1e2b02eb67ec)

## Cross-workspace data egress (consumer ws ≠ target ws)

df1 = spark.sql(f'''
SELECT shortcut_consumer_workspace, target_workspace_id,
       executingUPN, originatingApp,
       COUNT(*) AS ops
FROM {TBL['silver']}
WHERE accessStartTime > current_timestamp() - INTERVAL 7 DAY
  AND target_type = 'OneLake'
  AND shortcut_consumer_workspace IS NOT NULL
GROUP BY shortcut_consumer_workspace, target_workspace_id, executingUPN, originatingApp
ORDER BY ops DESC
''')
display(df1)

### User explicitly copies/saves shortcut data into a table or file

In [3]:
df1 = spark.sql(f'''
SELECT event_date, executingUPN, originatingApp, accessStartTime, operationName,
       shortcut_consumer_workspace, shortcut_consumer_lakehouse, shortcut_consumer_path,
       target_workspace_name, target_item_name, target_path, target_location, target_subpath
FROM gold_onelake_enriched_access
WHERE accessStartTime > current_timestamp() - INTERVAL 7 DAY
AND isShortcut=1
AND operationCategory='Write'
AND executingPrincipalType='User'
ORDER BY event_date, executingUPN
''')
display(df1)

StatementMeta(, 5b0df6fc-a3c0-49d4-97cf-0f1b2557d3a0, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 8ca60bd4-8d22-4521-b595-70eaa096a452)

## Suspicious external copies (PutBlobFromURL / CopyBlob via shortcut)

df2 = spark.sql(f'''
SELECT accessStartTime, executingUPN, callerIPAddress, originatingApp,
       operationName, shortcut_path_consumer, resolved_target_resource,
       target_type, target_location
FROM {TBL['silver']}
WHERE operationName IN ('PutBlobFromURL','CopyBlob','AbortCopyBlob')
  AND accessStartTime > current_timestamp() - INTERVAL 7 DAY
ORDER BY accessStartTime DESC
''')
display(df2)

In [1]:
df3 = spark.sql(f'''
SELECT accessStartTime, executingUPN, callerIPAddress, originatingApp,
       operationName, shortcut_consumer_workspace,  shortcut_consumer_lakehouse, 
       shortcut_consumer_path, target_type, target_location
FROM gold_onelake_enriched_access
WHERE operationName IN ('PutBlobFromURL','CopyBlob','AbortCopyBlob')
  AND accessStartTime > current_timestamp() - INTERVAL 7 DAY
ORDER BY accessStartTime DESC
''')
display(df3)

StatementMeta(, 5b0df6fc-a3c0-49d4-97cf-0f1b2557d3a0, 3, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 6b9d13c5-d9a4-4b2f-8646-29462ee2224e)

## External-tool access (azcopy, Storage Explorer, …) on any shortcut

In [5]:
spark.sql(f'''
SELECT accessStartTime, executingUPN, callerIPAddress, originatingApp,
       operationName, shortcut_path_consumer, resolved_target_resource
FROM {TBL['silver']}
WHERE accessStartTime > current_timestamp() - INTERVAL 1 DAY
  AND originatingApp RLIKE '(?i)(azcopy|storage.explorer|powershell|curl|python-requests|rclone)'
ORDER BY accessStartTime DESC
''').show(100, truncate=False)

StatementMeta(, b974943a-8831-4ea1-a732-8948035d231a, 7, Finished, Available, Finished, False)

+---------------+------------+---------------+--------------+-------------+----------------------+------------------------+
|accessStartTime|executingUPN|callerIPAddress|originatingApp|operationName|shortcut_path_consumer|resolved_target_resource|
+---------------+------------+---------------+--------------+-------------+----------------------+------------------------+
+---------------+------------+---------------+--------------+-------------+----------------------+------------------------+

